# 1. Data Preprocessing

This notebook handles:
- Missing value detection and imputation
- Duplicate row detection and removal
- Data type corrections (column names, datetimes, numerics)
- Dropping redundant columns

> **Input**: `predictive_maintenance_raw.csv`  
> **Output**: `predictive_maintenance_preprocessed.csv`

> **Next**: `02_eda.ipynb`


In [2]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 50)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Environment ready!')

ModuleNotFoundError: No module named 'plotly'

## 1. Load Raw Data


In [ ]:
# LOAD RAW DATA

df = pd.read_csv('predictive_maintenance_raw.csv')
print(f'Loaded: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nTotal NaN: {df.isnull().sum().sum()}')
df.head()

Loaded: (124594, 12)
Columns: ['date', 'device', 'failure', 'metric1', 'metric2', 'metric3', 'metric4', 'metric5', 'metric6', 'metric7', 'metric8', 'metric9']

Total NaN: 3414


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9
0,1/1/15,S1F01085,0,215630672,56,0,52,6,407438,0,0,7
1,1/1/15,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0
2,1/1/15,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0
3,1/1/15,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0
4,1/1/15,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3


## 2. Missing Value Analysis & Treatment


In [ ]:
# VISUALIZE MISSING PATTERNS

# Missing value counts
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

print('COLUMNS WITH MISSING VALUES')
print(missing_df)
print(f'\nTotal missing cells: {df.isnull().sum().sum()}')

COLUMNS WITH MISSING VALUES
         count  percent
metric4    413   0.3300
metric2    399   0.3200
metric1    392   0.3100
metric7    383   0.3100
metric3    377   0.3000
metric8    376   0.3000
metric6    369   0.3000
metric9    365   0.2900
metric5    340   0.2700

Total missing cells: 3414


In [ ]:
# Missing value counts
cols_with_missing = missing_df.index.tolist()

if cols_with_missing:
    fig = make_subplots(rows=1, cols=1, subplot_titles=('Missing Value Counts by Column', 'Missing Pattern Matrix (sample)'))

    # Bar chart of missing counts
    fig.add_trace(
        go.Bar(y=cols_with_missing, x=missing_df['count'], orientation='h', marker_color='coral', name='Counts'),
        row=1, col=1
    )

    fig.update_layout(height=500, showlegend=False)
    fig.show()

In [ ]:
# HANDLE STRING-ENCODED NULLS FIRST

# Before imputing NaN, we need to catch string representations of null
# Common patterns: 'N/A', 'null', 'missing', 'NA', 'n/a', '-', '#REF!', 'err'

null_strings = ['N/A', 'null', 'missing', 'NA', 'n/a', '-', '#REF!', 'err', 'None', '']

for col in df.columns:
    if df[col].dtype == object:
        mask = df[col].isin(null_strings)
        if mask.sum() > 0:
            print(f'{col}: {mask.sum()} string-null values found → converting to NaN')
            df.loc[mask, col] = np.nan

print(f'\nTotal NaN after string-null conversion: {df.isnull().sum().sum()}')

metric1: 244 string-null values found → converting to NaN
metric2: 242 string-null values found → converting to NaN
metric3: 260 string-null values found → converting to NaN
metric4: 236 string-null values found → converting to NaN
metric5: 224 string-null values found → converting to NaN
metric6: 230 string-null values found → converting to NaN
metric7: 261 string-null values found → converting to NaN
metric8: 242 string-null values found → converting to NaN
metric9: 263 string-null values found → converting to NaN

Total NaN after string-null conversion: 5616


In [ ]:
# 1. Sort records chronologically by device and date
df['date'] = pd.to_datetime(df['date'], format='mixed')
df = df.sort_values(by=['device', 'date']).reset_index(drop=True)

# Define column groups
group_a_cols = ['metric1', 'metric5', 'metric6']  # Continuous sensors
group_b_cols = ['metric2', 'metric3', 'metric4', 'metric7', 'metric8', 'metric9']  # Error/count logs
all_metrics = group_a_cols + group_b_cols

# Convert metric columns from object dtype to numeric
df[all_metrics] = df[all_metrics].apply(pd.to_numeric, errors='coerce')

# 2. Forward fill per device (only filling NaN values)
df[all_metrics] = df[all_metrics].fillna(df.groupby('device')[all_metrics].ffill())

# 3. Backward fill Group A (only filling remaining initial NaNs per device)
df[group_a_cols] = df[group_a_cols].fillna(df.groupby('device')[group_a_cols].bfill())

# 4. Zero fill Group B for initial missing values per device
df[group_b_cols] = df[group_b_cols].fillna(0)

# 5. Final Fallback: Fill any remaining NaNs (e.g., devices with no records at all)
df[group_a_cols] = df[group_a_cols].fillna(df[group_a_cols].median())

In [ ]:
#cross checking
print("Remaining missing values:", df[group_a_cols + group_b_cols].isnull().sum().sum())

Remaining missing values: 0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124594 entries, 0 to 124593
Data columns (total 12 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   date     124594 non-null  datetime64[ns]
 1   device   124594 non-null  object        
 2   failure  124594 non-null  object        
 3   metric1  124594 non-null  float64       
 4   metric2  124594 non-null  int64         
 5   metric3  124594 non-null  int64         
 6   metric4  124594 non-null  int64         
 7   metric5  124594 non-null  int64         
 8   metric6  124594 non-null  float64       
 9   metric7  124594 non-null  int64         
 10  metric8  124594 non-null  int64         
 11  metric9  124594 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(7), object(2)
memory usage: 11.4+ MB


## 3. Duplicate Detection & Removal


In [ ]:

cols_to_check = [col for col in df.columns if col not in ['id', 'Unnamed: 0']]
# Find duplicate rows (keep='first' marks all occurrences after the 1st as True)
duplicate_mask = df.duplicated(subset=cols_to_check, keep='first')
num_duplicates = duplicate_mask.sum()

print("=" * 40)
print("DUPLICATE DETECTION SUMMARY")
print("=" * 40)
print(f"Total Rows Before Deduplication : {len(df)}")
print(f"Total Duplicate Rows Found      : {num_duplicates}")
# 3. INSPECT DUPLICATE ROWS (OPTIONAL)
if num_duplicates > 0:
    print("\nSample Duplicate Rows (showing non-first occurrences):")
    display(df[duplicate_mask].head())
else:
    print("\nNo duplicate rows detected.")
# 4. REMOVE DUPLICATE ROWS
# Drop duplicates and reset dataframe index
df = df.drop_duplicates(subset=cols_to_check, keep='first').reset_index(drop=True)

print("\n" + "=" * 40)
print("DEDUPLICATION COMPLETE")
print("=" * 40)
print(f"Final Row Count After Deduplication: {len(df)}")
print(f"Remaining Duplicates: {df.duplicated(subset=cols_to_check).sum()}")

DUPLICATE DETECTION SUMMARY
Total Rows Before Deduplication : 124594
Total Duplicate Rows Found      : 101

Sample Duplicate Rows (showing non-first occurrences):


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9
327,2015-02-24,S1F13GPY,0,179889312.0000,0,0,0,16,327463.0000,0,0,0
3193,2015-07-14,S1F0BVK1,0,116267320.0000,0,0,0,6,262224.0000,8,8,0
3570,2015-03-18,S1F0C95J,0,124110968.0000,0,0,0,5,256402.0000,0,0,0
3947,2015-02-10,S1F0CVWK,0,200781960.0000,1960,0,6,8,371858.0000,0,0,0
4862,2015-01-15,S1F0EGMT,0,134892016.0000,0,0,0,9,215198.0000,0,0,0



DEDUPLICATION COMPLETE
Final Row Count After Deduplication: 124493
Remaining Duplicates: 0


## 4. Data Type Correction & Parsing


In [ ]:

date_cols = ['date']  # Add extra date columns if present, e.g., ['date', 'timestamp']
# DATETIME CLEANING & PARSING PIPELINE

for col in date_cols:
    if col in df.columns:
        # Step 1: Convert to string and strip leading/trailing whitespace
        s = df[col].astype(str).str.strip()

        # Step 2: Remove non-breaking spaces (\xa0) & multiple spaces
        s = s.str.replace('\xa0', ' ', regex=False).str.replace(r'\s+', ' ', regex=True)

        # Step 3: Strip timezone suffixes (EST, UTC, PST, CST, GMT, etc.)
        s = s.str.replace(r'\s*(?:EST|PST|CST|MST|UTC|GMT)$', '', regex=True, case=False)

        # Step 4: Parse dates with mixed format support
        # errors='coerce' turns unparseable/corrupted dates into NaT
        df[col] = pd.to_datetime(s, format='mixed', errors='coerce')


# DROP CORRUPTED / INVALID DATETIME ROWS
initial_count = len(df)

# Drop rows where any specified date column resulted in NaT
df = df.dropna(subset=date_cols).reset_index(drop=True)

final_count = len(df)
dropped_count = initial_count - final_count

print(f"Successfully processed datetime column(s): {date_cols}")
print(f"Rows dropped due to corrupted/unparseable dates: {dropped_count}")
print(f"Remaining clean rows: {final_count}")

Successfully processed datetime column(s): ['date']
Rows dropped due to corrupted/unparseable dates: 0
Remaining clean rows: 124493


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124493 entries, 0 to 124492
Data columns (total 12 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   date     124493 non-null  datetime64[ns]
 1   device   124493 non-null  object        
 2   failure  124493 non-null  object        
 3   metric1  124493 non-null  float64       
 4   metric2  124493 non-null  int64         
 5   metric3  124493 non-null  int64         
 6   metric4  124493 non-null  int64         
 7   metric5  124493 non-null  int64         
 8   metric6  124493 non-null  float64       
 9   metric7  124493 non-null  int64         
 10  metric8  124493 non-null  int64         
 11  metric9  124493 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(7), object(2)
memory usage: 11.4+ MB


In [ ]:
df["failure"].unique()

array([0, 1])

In [ ]:
# 1. Convert to string, strip extra whitespace, and make lowercase
s = df['failure'].astype(str).str.strip().str.lower()

# 2. Map '1' to integer 1, and convert all non-failure variants ('0', '0.0', 'false', '0 ', etc.) to 0
df['failure'] = s.map(lambda x: 1 if x in ['1', '1.0', 'true'] else 0).astype(int)

# Verify the result
print("Unique values in 'failure' column:", df['failure'].unique())
print("\nValue Counts:")
print(df['failure'].value_counts())

Unique values in 'failure' column: [0 1]

Value Counts:
failure
0    124387
1       106
Name: count, dtype: int64


In [ ]:
print('\nData types after fix:')
print(df.dtypes)


Data types after fix:
date       datetime64[ns]
device             object
failure             int64
metric1           float64
metric2             int64
metric3             int64
metric4             int64
metric5             int64
metric6           float64
metric7             int64
metric8             int64
metric9             int64
dtype: object


## DROP REDUNDANT COLUMNS

In [ ]:
# DROP REDUNDANT COLUMNS
# Remove redundant duplicate column
if 'metric8' in df.columns:
    df = df.drop(columns=['metric8'])
    print("Dropped redundant column 'metric8' (100% identical to 'metric7').")

Dropped redundant column 'metric8' (100% identical to 'metric7').


## 5. Save Preprocessed Data


In [ ]:
# SAVE PREPROCESSED DATASET

print(f'Final shape: {df.shape}')
print(f'Remaining NaN: {df.isnull().sum().sum()}')
print(f'Columns: {df.columns.tolist()}')
print(f'Data types:\n{df.dtypes}')

df.to_csv('predictive_maintenance_preprocessed.csv', index=False)
print('\nSaved: predictive_maintenance_preprocessed.csv')

Final shape: (124493, 11)
Remaining NaN: 0
Columns: ['date', 'device', 'failure', 'metric1', 'metric2', 'metric3', 'metric4', 'metric5', 'metric6', 'metric7', 'metric9']
Data types:
date       datetime64[ns]
device             object
failure             int64
metric1           float64
metric2             int64
metric3             int64
metric4             int64
metric5             int64
metric6           float64
metric7             int64
metric9             int64
dtype: object

Saved: predictive_maintenance_preprocessed.csv
